# Pneumonia Diagnosis, Detection & Localization - full pipeline (Kaggle)

**UE24CS352A Machine Learning - Mini Project** · Samvith Vaddiparthi (PES1UG24AM468) · Sanjit C (PES1UG24AM448)

Re-implements and extends *Weakly Supervised Pneumonia Localization* (Stanford CS229, 2018):
classical baselines (LR / SVM / RF) → the paper's 10-layer CNN + CAM → ImageNet-pretrained DenseNet-121 + CAM →
supervised Faster R-CNN → classifier-gated detector.

### Before you press *Run All*
1. **Add data**: right panel → *Add Input* → search **RSNA Pneumonia Detection Challenge** (competition) → Add.
   (You must have clicked *Join Competition / I understand and accept* on the competition page once.)
2. **Settings → Accelerator → GPU T4 x1** (or P100).
3. **Settings → Internet → On** (downloads ImageNet / COCO weights once; needs a phone-verified Kaggle account).
4. Choose `MODE` in the next cell: full run ≈ 2-3 h, `--fast` ≈ 45 min.
5. Best: **Save Version → Save & Run All (Commit)** - it keeps running when you close the browser.
   Afterwards download `results_light.zip` and `model_weights.zip` from the *Output* tab.

In [ ]:
MODE = ""          # "" = full configuration (recommended);  "--fast" = fewer epochs (~45 min)
import os, glob, subprocess
cands = glob.glob("/kaggle/input/**/stage_2_train_labels.csv", recursive=True)
assert cands, "RSNA dataset not attached - see step 1 above"
DATA_DIR = os.path.dirname(cands[0])
os.environ["DATA_DIR"], os.environ["MODE"] = DATA_DIR, MODE
print("data:", DATA_DIR)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True,
                     capture_output=True, text=True).stdout or "NO GPU - enable it in Settings!")

In [ ]:
!pip -q install pydicom pylibjpeg pylibjpeg-libjpeg
import urllib.request
try:
    urllib.request.urlopen("https://download.pytorch.org", timeout=10); print("internet OK")
except Exception as e:
    print("WARNING: no internet - pretrained weights cannot be downloaded. Turn Internet ON.", e)

## 1 · Write the project source code
(identical to the GitHub repository)

In [ ]:
!mkdir -p /kaggle/working/repo/src/pneumo /kaggle/working/repo/scripts

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/__init__.py
"""Pneumonia diagnosis, detection and localization on chest X-rays (RSNA)."""
__version__ = "1.0.0"

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/baselines.py
"""Classical ML baselines on flattened pixels (as in the Stanford report).

LR and SVM work on a PCA projection of the standardised pixels (4096 raw
features is more than the classical models can use well); Random Forest uses
the raw pixels directly.
"""
from __future__ import annotations

import cv2
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def flatten_images(arr, idx, size: int) -> np.ndarray:
    out = np.empty((len(idx), size * size), dtype=np.float32)
    for j, i in enumerate(idx):
        out[j] = cv2.resize(np.asarray(arr[i]), (size, size), interpolation=cv2.INTER_AREA).ravel() / 255.0
    return out


def build_baselines(n_pca: int, seed: int = 42) -> dict:
    return {
        "LogReg": make_pipeline(StandardScaler(), PCA(n_pca, random_state=seed),
                                LogisticRegression(max_iter=2000, C=1.0)),
        "SVM": make_pipeline(StandardScaler(), PCA(n_pca, random_state=seed),
                             SVC(kernel="rbf", C=1.0, gamma="scale", probability=False)),
        "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1,
                                               random_state=seed),
    }


def scores(model, X) -> np.ndarray:
    """Probability-like score in [0, 1] (SVM margin squashed with a sigmoid)."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    d = model.decision_function(X)
    return 1 / (1 + np.exp(-d))

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/cam.py
"""Weakly-supervised localization: Class Activation Map -> bounding boxes.

Pipeline (as in the Stanford report, with two changes noted below):
  1. CAM  M(x, y) = sum_k w_k f_k(x, y) from the last conv map and FC weights.
  2. ReLU, bilinear up-sampling to image size, normalise by the map's maximum
     (a *relative* threshold -- the report used a fixed cut-off and noted that
     it should be dynamic).
  3. Threshold and find connected blobs.  The report used a hand-written DFS
     flood fill; `scipy.ndimage.label` computes exactly the same connected
     components, much faster.
  4. One box per blob (min/max x, y).  Tiny blobs are dropped and at most
     `max_boxes` are kept (RSNA images have at most 4 boxes).
The threshold is tuned on the validation set instead of fixed by hand.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage
from torch.utils.data import DataLoader


@torch.no_grad()
def compute_cams(model, ds, device, batch: int = 32, workers: int = 2):
    """Returns probs (N,) and low-resolution CAMs (N, h, w) as float16."""
    model.eval().to(device)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers)
    probs, cams = [], []
    for x, _ in dl:
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            logit, cam = model.cam(x.to(device))
        probs.append(torch.sigmoid(logit.float()).cpu())
        cams.append(cam.float().cpu().half())
    return torch.cat(probs).numpy(), torch.cat(cams).numpy()


def upsample(cam: np.ndarray, h: int, w: int) -> np.ndarray:
    t = torch.from_numpy(cam.astype(np.float32))[None, None]
    return F.interpolate(t, size=(h, w), mode="bilinear", align_corners=False)[0, 0].numpy()


def cam_to_boxes(cam: np.ndarray, out_h: int, out_w: int, thr: float, min_area_frac: float = 0.01,
                 max_boxes: int = 3, work_size: int = 256):
    """CAM (h, w) -> list of boxes (x1, y1, x2, y2) in out_h x out_w pixels + blob scores."""
    m = np.maximum(upsample(cam, work_size, work_size), 0)
    peak = m.max()
    if peak <= 0:
        return [], []
    m = m / peak
    lab, n = ndimage.label(m >= thr)
    boxes, scores = [], []
    if n == 0:
        return boxes, scores
    objs = ndimage.find_objects(lab)
    sx, sy = out_w / work_size, out_h / work_size
    for k, sl in enumerate(objs, start=1):
        area = (lab[sl] == k).sum()
        if area < min_area_frac * work_size * work_size:
            continue
        ys, xs = sl
        boxes.append([xs.start * sx, ys.start * sy, xs.stop * sx, ys.stop * sy])
        scores.append(float(m[sl][lab[sl] == k].mean()))
    order = np.argsort(scores)[::-1][:max_boxes]
    return [boxes[i] for i in order], [scores[i] for i in order]


def cam_predictions(probs, cams, sizes, thr: float, gate: float | None, min_area_frac: float,
                    max_boxes: int):
    """Turn CAMs into per-image predictions.  If `gate` is given, images whose
    classifier probability is below it get no box (end-to-end mode, as in the
    report); `gate=None` localizes every image (the report's 'oracle' mode)."""
    preds = []
    for p, cam, (h, w) in zip(probs, cams, sizes):
        if gate is not None and p < gate:
            preds.append(([], []))
            continue
        b, s = cam_to_boxes(cam, h, w, thr, min_area_frac, max_boxes)
        preds.append((b, [float(p) * si for si in s]))
    return preds

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/config.py
"""Central configuration for every stage of the pipeline.

All hyper-parameters live here so that the README, the write-up and the code
agree. `Config.fast()` returns a reduced configuration (fewer epochs) for a
quick end-to-end run; `Config.smoke()` is a tiny CPU configuration used by the
synthetic-data smoke test.
"""
from __future__ import annotations

import dataclasses
import json
from dataclasses import dataclass, field
from pathlib import Path


@dataclass
class Config:
    # ---------------- paths ----------------
    data_dir: str = "/kaggle/input/rsna-pneumonia-detection-challenge"
    cache_dir: str = "cache"          # pre-processed image arrays (not committed)
    out_dir: str = "outputs"          # results.json, figures, model weights

    # ---------------- data ----------------
    cache_size: int = 512             # every DICOM is resized once to this size
    split_fracs: tuple = (0.70, 0.15, 0.15)   # train / val / test, patient level
    seed: int = 42
    limit: int | None = None          # use only N images (debugging)
    num_workers: int = 4

    # ---------------- classical baselines (LR / SVM / RF) ----------------
    baseline_size: int = 64           # images flattened at 64x64 = 4096 features
    baseline_pca: int = 150
    baseline_max_train: int = 12000   # SVM is O(n^2); cap the train set

    # ---------------- paper-style CNN (replication) ----------------
    paper_cnn_size: int = 128         # the paper compressed 1024 -> 128
    paper_cnn_epochs: int = 20        # paper: Adam, lr 1e-4, 20 epochs
    paper_cnn_lr: float = 1e-4
    paper_cnn_batch: int = 64

    # ---------------- DenseNet-121 transfer learning (improvement) ----------------
    densenet_size: int = 384
    densenet_epochs: int = 10
    densenet_lr: float = 3e-4
    densenet_batch: int = 32
    pretrained: bool = True           # ImageNet weights (needs internet once)

    # ---------------- CAM -> boxes ----------------
    cam_thresholds: tuple = (0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
    cam_min_area_frac: float = 0.01   # drop blobs smaller than 1% of the image
    cam_max_boxes: int = 3

    # ---------------- Faster R-CNN (supervised localization) ----------------
    det_size: int = 512
    det_epochs: int = 6
    det_lr: float = 0.01
    det_batch: int = 8
    det_neg_ratio: float = 1.0        # negatives sampled per positive each epoch
    det_score_thresholds: tuple = (0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
    gate_thresholds: tuple = (0.2, 0.3, 0.4, 0.5, 0.6)

    # which experiments to run
    settings: tuple = ("paper", "full")
    run_detector: bool = True

    # ------------------------------------------------------------------
    @staticmethod
    def fast(**kw) -> "Config":
        """~45 min on a Kaggle T4: fewer epochs, same models."""
        base = dict(paper_cnn_epochs=8, densenet_epochs=4, densenet_size=320, det_epochs=2)
        return Config(**{**base, **kw})

    @staticmethod
    def smoke(**kw) -> "Config":
        """Tiny CPU config for the synthetic-data smoke test."""
        base = dict(cache_size=128, baseline_size=32, baseline_pca=10,
                    paper_cnn_size=64, paper_cnn_epochs=2, paper_cnn_batch=16,
                    densenet_size=96, densenet_epochs=1, densenet_batch=8,
                    det_size=128, det_epochs=1, det_batch=4, pretrained=False, num_workers=0)
        return Config(**{**base, **kw})

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.to_dict(), indent=2))

    @staticmethod
    def load(path: str | Path) -> "Config":
        d = json.loads(Path(path).read_text())
        for k, v in list(d.items()):
            if isinstance(v, list):
                d[k] = tuple(v)
        return Config(**d)

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/data.py
"""Data loading for the RSNA Pneumonia Detection Challenge.

Files expected in `data_dir` (as provided by Kaggle):
    stage_2_train_labels.csv        patientId, x, y, width, height, Target
    stage_2_detailed_class_info.csv patientId, class
    stage_2_train_images/<patientId>.dcm

One row of `stage_2_train_labels.csv` is one bounding box, so a patient with
two opacities has two rows. We collapse these into one row per image with a
list of boxes, split *by patient* (one image per patient, so no leakage), and
decode every DICOM exactly once into a uint8 memory-mapped array.
"""
from __future__ import annotations

import json
from functools import partial
from multiprocessing import Pool
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

CLASS_NORMAL = "Normal"
CLASS_OPACITY = "Lung Opacity"
CLASS_OTHER = "No Lung Opacity / Not Normal"
CLASSES = [CLASS_NORMAL, CLASS_OTHER, CLASS_OPACITY]


# ----------------------------------------------------------------------------
# label table
# ----------------------------------------------------------------------------
def build_label_table(data_dir: str | Path) -> pd.DataFrame:
    """One row per image: patientId, target, cls, boxes (list of [x, y, w, h])."""
    data_dir = Path(data_dir)
    labels = pd.read_csv(data_dir / "stage_2_train_labels.csv")
    info = pd.read_csv(data_dir / "stage_2_detailed_class_info.csv").drop_duplicates("patientId")

    def _boxes(g: pd.DataFrame) -> list:
        g = g.dropna(subset=["x"])
        return g[["x", "y", "width", "height"]].astype(float).values.tolist()

    rows = []
    for pid, g in labels.groupby("patientId", sort=False):
        rows.append({"patientId": pid, "target": int(g["Target"].max()), "boxes": _boxes(g)})
    df = pd.DataFrame(rows).merge(info, on="patientId", how="left")
    df = df.rename(columns={"class": "cls"})
    df["n_boxes"] = df["boxes"].apply(len)
    return df.reset_index(drop=True)


def make_split(df: pd.DataFrame, fracs=(0.7, 0.15, 0.15), seed: int = 42) -> pd.DataFrame:
    """Stratified (by 3-way class) patient-level train/val/test split."""
    tr, va, te = fracs
    idx = np.arange(len(df))
    i_tr, i_rest = train_test_split(idx, train_size=tr, stratify=df["cls"], random_state=seed)
    rel = va / (va + te)
    i_va, i_te = train_test_split(i_rest, train_size=rel, stratify=df["cls"].iloc[i_rest],
                                  random_state=seed)
    split = np.empty(len(df), dtype=object)
    split[i_tr], split[i_va], split[i_te] = "train", "val", "test"
    df = df.copy()
    df["split"] = split
    return df


def setting_subset(df: pd.DataFrame, setting: str) -> pd.DataFrame:
    """`paper`: Normal vs Lung Opacity only (the Stanford report removed the
    'No Lung Opacity / Not Normal' images).  `full`: all images; positive =
    Lung Opacity, negative = everything else (the real, harder task)."""
    if setting == "paper":
        return df[df["cls"] != CLASS_OTHER]
    if setting == "full":
        return df
    raise ValueError(setting)


# ----------------------------------------------------------------------------
# DICOM decoding + caching
# ----------------------------------------------------------------------------
def read_dicom(path: str | Path) -> np.ndarray:
    """Return a uint8 2-D array (0 = black) from a DICOM file."""
    import pydicom
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        img = img.max() - img
    lo, hi = float(img.min()), float(img.max())
    img = (img - lo) / max(hi - lo, 1e-6) * 255.0
    return img.astype(np.uint8)


def read_any_image(path: str | Path) -> np.ndarray:
    """DICOM / PNG / JPG -> uint8 grayscale (used by the demo and predict.py)."""
    path = str(path)
    if path.lower().endswith(".dcm"):
        return read_dicom(path)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"cannot read image {path}")
    return img


def _load_resize(pid: str, img_dir: str, size: int):
    img = read_dicom(Path(img_dir) / f"{pid}.dcm")
    h, w = img.shape
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA), h, w


def build_cache(df: pd.DataFrame, data_dir: str | Path, cache_dir: str | Path,
                size: int = 512, workers: int = 4) -> pd.DataFrame:
    """Decode all DICOMs once into `cache_dir/img{size}.npy` (memmap, N x size x size).

    Adds columns `idx` (row in the cache), `orig_h`, `orig_w` and writes
    `cache_dir/meta.csv`.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    img_dir = Path(data_dir) / "stage_2_train_images"
    arr = np.lib.format.open_memmap(cache_dir / f"img{size}.npy", mode="w+",
                                    dtype=np.uint8, shape=(len(df), size, size))
    hs, ws = np.zeros(len(df), int), np.zeros(len(df), int)
    fn = partial(_load_resize, img_dir=str(img_dir), size=size)
    pids = df["patientId"].tolist()
    if workers > 1:
        with Pool(workers) as pool:
            it = pool.imap(fn, pids, chunksize=32)
            for i, (img, h, w) in enumerate(it):
                arr[i], hs[i], ws[i] = img, h, w
                if i % 2000 == 0:
                    print(f"  cached {i}/{len(df)}", flush=True)
    else:
        for i, pid in enumerate(pids):
            arr[i], hs[i], ws[i] = fn(pid)
    arr.flush()
    df = df.copy()
    df["idx"], df["orig_h"], df["orig_w"] = np.arange(len(df)), hs, ws
    out = df.copy()
    out["boxes"] = out["boxes"].apply(json.dumps)
    out.to_csv(cache_dir / "meta.csv", index=False)
    return df


def load_meta(cache_dir: str | Path) -> pd.DataFrame:
    df = pd.read_csv(Path(cache_dir) / "meta.csv")
    df["boxes"] = df["boxes"].apply(json.loads)
    return df


def load_cache(cache_dir: str | Path, size: int) -> np.ndarray:
    return np.load(Path(cache_dir) / f"img{size}.npy", mmap_mode="r")


# ----------------------------------------------------------------------------
# augmentation (numpy / OpenCV, so it works in DataLoader workers)
# ----------------------------------------------------------------------------
def random_affine(img: np.ndarray, rng: np.random.Generator, rot=10, scale=(0.9, 1.1),
                  shift=0.06) -> np.ndarray:
    h, w = img.shape
    ang = rng.uniform(-rot, rot)
    sc = rng.uniform(*scale)
    m = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    m[:, 2] += rng.uniform(-shift, shift, 2) * np.array([w, h])
    return cv2.warpAffine(img, m, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=0)


def random_intensity(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    a = rng.uniform(0.9, 1.1)          # contrast
    b = rng.uniform(-12, 12)           # brightness
    return np.clip(img.astype(np.float32) * a + b, 0, 255).astype(np.uint8)


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def to_tensor(img: np.ndarray, three_channel: bool):
    """uint8 HxW -> float tensor. 1-ch: (x-mean)/std per image. 3-ch: ImageNet norm."""
    import torch
    x = img.astype(np.float32) / 255.0
    if three_channel:
        x = np.repeat(x[None], 3, 0)
        x = (x - IMAGENET_MEAN[:, None, None]) / IMAGENET_STD[:, None, None]
    else:
        x = ((x - x.mean()) / (x.std() + 1e-6))[None]
    return torch.from_numpy(np.ascontiguousarray(x))


# ----------------------------------------------------------------------------
# torch datasets
# ----------------------------------------------------------------------------
class ClsDataset:
    """Image-level classification dataset (pneumonia vs not)."""

    def __init__(self, df: pd.DataFrame, cache_dir, cache_size: int, size: int,
                 three_channel: bool, augment: bool, seed: int = 0):
        self.idx = df["idx"].values
        self.y = df["target"].values.astype(np.float32)
        self.cache_dir, self.cache_size = cache_dir, cache_size
        self.size, self.three, self.augment = size, three_channel, augment
        self._arr = None
        self.seed = seed

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        import torch
        if self._arr is None:
            self._arr = load_cache(self.cache_dir, self.cache_size)
        img = np.asarray(self._arr[self.idx[i]])
        if self.size != self.cache_size:
            img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA)
        if self.augment:
            rng = np.random.default_rng()
            img = random_affine(img, rng)
            img = random_intensity(img, rng)
            if rng.random() < 0.5:
                img = img[:, ::-1]
        return to_tensor(img, self.three), torch.tensor(self.y[i])


class DetDataset:
    """Detection dataset for torchvision Faster R-CNN (boxes in cache pixels)."""

    def __init__(self, df: pd.DataFrame, cache_dir, cache_size: int, size: int, augment: bool):
        self.df = df.reset_index(drop=True)
        self.cache_dir, self.cache_size, self.size, self.augment = cache_dir, cache_size, size, augment
        self._arr = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        import torch
        if self._arr is None:
            self._arr = load_cache(self.cache_dir, self.cache_size)
        r = self.df.iloc[i]
        img = np.asarray(self._arr[r["idx"]])
        if self.size != self.cache_size:
            img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA)
        sx, sy = self.size / r["orig_w"], self.size / r["orig_h"]
        boxes = np.array([[x * sx, y * sy, (x + w) * sx, (y + h) * sy] for x, y, w, h in r["boxes"]],
                         dtype=np.float32).reshape(-1, 4)
        if self.augment:
            rng = np.random.default_rng()
            img = random_intensity(img, rng)
            if rng.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1])
                if len(boxes):
                    boxes[:, [0, 2]] = self.size - boxes[:, [2, 0]]
        x = torch.from_numpy(np.repeat((img.astype(np.float32) / 255.0)[None], 3, 0))
        target = {"boxes": torch.from_numpy(boxes),
                  "labels": torch.ones(len(boxes), dtype=torch.int64)}
        return x, target, int(i)


def det_collate(batch):
    return tuple(zip(*batch))

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/infer.py
"""Inference on a single X-ray: used by predict.py (CLI) and app.py (demo)."""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
import torch

from .cam import cam_to_boxes, upsample
from .data import to_tensor
from .models import build_classifier, build_detector
from .utils import get_device, load_json


class PneumoniaPredictor:
    """Loads the trained DenseNet-121 (+ optional Faster R-CNN) and the
    thresholds that were tuned on the validation set (from results.json)."""

    def __init__(self, out_dir: str | Path = "outputs", setting: str = "full", device=None):
        out_dir = Path(out_dir)
        self.device = device or get_device()
        res = load_json(out_dir / "results.json")
        cfg = res.get("config", {})
        self.cls_size = int(cfg.get("densenet_size", 384))
        self.det_size = int(cfg.get("det_size", 512))
        loc = res.get("localization", {}).get(setting, {})
        cam_cfg = loc.get("DenseNet121+CAM", {})
        self.cls_thr = float(cam_cfg.get("cls_gate",
                             res.get("classification", {}).get(setting, {}).get("DenseNet121", {})
                             .get("threshold", 0.5)))
        self.cam_thr = float(cam_cfg.get("cam_threshold", 0.5))
        gated = loc.get("DenseNet-gated FasterRCNN", {})
        self.det_thr = float(gated.get("score_threshold", loc.get("FasterRCNN", {}).get("score_threshold", 0.5)))
        self.min_area = float(cfg.get("cam_min_area_frac", 0.01))

        w = out_dir / "models" / f"DenseNet121_{setting}.pt"
        if not w.exists():
            raise FileNotFoundError(f"{w} not found - run training first or copy the weights there")
        self.clf = build_classifier("DenseNet121", pretrained=False)
        self.clf.load_state_dict(torch.load(w, map_location="cpu"))
        self.clf.eval().to(self.device)
        self.det = None
        dw = out_dir / "models" / "FasterRCNN.pt"
        if dw.exists():
            self.det = build_detector(pretrained=False, size=self.det_size)
            self.det.load_state_dict(torch.load(dw, map_location="cpu"))
            self.det.eval().to(self.device)

    @torch.no_grad()
    def __call__(self, img: np.ndarray) -> dict:
        """img: uint8 grayscale, any size. Boxes are returned in its pixels."""
        h, w = img.shape[:2]
        x = cv2.resize(img, (self.cls_size, self.cls_size), interpolation=cv2.INTER_AREA)
        logit, cam = self.clf.cam(to_tensor(x, True)[None].to(self.device))
        prob = float(torch.sigmoid(logit)[0])
        cam = cam[0].float().cpu().numpy()
        cam_boxes, cam_scores = cam_to_boxes(cam, h, w, self.cam_thr, self.min_area, 3)
        heat = np.maximum(upsample(cam, h, w), 0)
        heat = heat / heat.max() if heat.max() > 0 else heat
        out = {"prob": prob, "positive": prob >= self.cls_thr, "cls_threshold": self.cls_thr,
               "heatmap": heat, "cam_boxes": cam_boxes if prob >= self.cls_thr else [],
               "det_boxes": [], "det_scores": []}
        if self.det is not None:
            xd = cv2.resize(img, (self.det_size, self.det_size), interpolation=cv2.INTER_AREA)
            t = torch.from_numpy(np.repeat((xd.astype(np.float32) / 255)[None], 3, 0)).to(self.device)
            r = self.det([t])[0]
            b, s = r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()
            keep = s >= self.det_thr
            b = b[keep] * np.array([w / self.det_size, h / self.det_size] * 2)
            if prob >= self.cls_thr:
                out["det_boxes"], out["det_scores"] = b.tolist(), s[keep].tolist()
        return out


def render(img: np.ndarray, pred: dict, gt_boxes=None) -> np.ndarray:
    """RGB visualisation: CAM heat-map, CAM boxes (blue), detector boxes (green),
    ground truth (red, optional)."""
    base = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    heat = cv2.applyColorMap((pred["heatmap"] * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    vis = cv2.addWeighted(base, 0.65, heat, 0.35, 0)
    t = max(2, img.shape[0] // 250)
    for b in gt_boxes or []:
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (227, 52, 47), t)
    for b in pred["cam_boxes"]:
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (47, 111, 227), t)
    for b, s in zip(pred["det_boxes"], pred["det_scores"]):
        cv2.rectangle(vis, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (24, 165, 88), t)
        cv2.putText(vis, f"{s:.2f}", (int(b[0]), max(int(b[1]) - 6, 12)), cv2.FONT_HERSHEY_SIMPLEX,
                    img.shape[0] / 1400, (24, 165, 88), t)
    return vis

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/metrics.py
"""Evaluation metrics.

Classification : accuracy, precision, recall (sensitivity), specificity, F1,
                 ROC-AUC, confusion matrix.
Localization   : (1) mean IoU as in the Stanford report: for every pneumonia
                     image, IoU between the union of predicted boxes and the
                     union of ground-truth boxes; an image with no predicted box
                     scores 0.
                 (2) the official RSNA Kaggle metric: mean over images of the
                     average (over IoU thresholds 0.40:0.05:0.75) of
                     TP / (TP + FP + FN).

All boxes are (x1, y1, x2, y2) in original-image pixels.
"""
from __future__ import annotations

import numpy as np
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve

RSNA_THRESHOLDS = np.arange(0.40, 0.76, 0.05)


# ---------------------------------------------------------------- classification
def classification_metrics(y_true, prob, thr: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    try:
        auc = roc_auc_score(y_true, prob)
    except ValueError:
        auc = float("nan")
    return {"accuracy": (tp + tn) / len(y_true), "precision": prec, "recall": rec,
            "specificity": spec, "f1": f1, "auc": auc, "threshold": thr,
            "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}}


def best_f1_threshold(y_true, prob) -> float:
    """Threshold maximising F1 on (validation) data."""
    y_true = np.asarray(y_true).astype(int)
    grid = np.linspace(0.05, 0.95, 91)
    f1 = np.array([classification_metrics(y_true, prob, t)["f1"] for t in grid])
    ties = grid[f1 >= f1.max() - 1e-9]
    return float(np.median(ties))          # middle of the best range (robust to ties)


def roc_points(y_true, prob, n: int = 200):
    fpr, tpr, _ = roc_curve(y_true, prob)
    grid = np.linspace(0, 1, n)
    return grid.tolist(), np.interp(grid, fpr, tpr).tolist()


# ---------------------------------------------------------------- boxes
def xywh_to_xyxy(boxes):
    return [[x, y, x + w, y + h] for x, y, w, h in boxes]


def box_iou(a, b) -> float:
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def union_mask_iou(pred, gt, h: int = 1024, w: int = 1024) -> float:
    """IoU between the union of predicted boxes and the union of GT boxes."""
    if len(gt) == 0 and len(pred) == 0:
        return 1.0
    if len(pred) == 0 or len(gt) == 0:
        return 0.0
    mp = np.zeros((h, w), bool)
    mg = np.zeros((h, w), bool)
    for x1, y1, x2, y2 in pred:
        mp[max(int(y1), 0):max(int(round(y2)), 0), max(int(x1), 0):max(int(round(x2)), 0)] = True
    for x1, y1, x2, y2 in gt:
        mg[max(int(y1), 0):int(round(y2)), max(int(x1), 0):int(round(x2))] = True
    inter = np.logical_and(mp, mg).sum()
    union = np.logical_or(mp, mg).sum()
    return float(inter / union) if union else 0.0


def rsna_image_score(pred_boxes, pred_scores, gt_boxes):
    """Kaggle RSNA metric for one image; returns None if the image is ignored
    (no ground truth and no prediction)."""
    if len(gt_boxes) == 0 and len(pred_boxes) == 0:
        return None
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return 0.0
    order = np.argsort(-np.asarray(pred_scores))
    preds = [pred_boxes[i] for i in order]
    ious = np.array([[box_iou(p, g) for g in gt_boxes] for p in preds])
    scores = []
    for t in RSNA_THRESHOLDS:
        matched = set()
        tp = 0
        for pi in range(len(preds)):
            best_j, best_iou = -1, t
            for gj in range(len(gt_boxes)):
                if gj not in matched and ious[pi, gj] > best_iou:
                    best_j, best_iou = gj, ious[pi, gj]
            if best_j >= 0:
                matched.add(best_j)
                tp += 1
        fp = len(preds) - tp
        fn = len(gt_boxes) - tp
        scores.append(tp / (tp + fp + fn))
    return float(np.mean(scores))


def localization_metrics(preds: list, gts: list, sizes: list) -> dict:
    """preds: list of (boxes, scores) per image; gts: list of GT box lists;
    sizes: list of (h, w).  Returns mean IoU (over positive images) and RSNA mAP
    (over all images that are not ignored)."""
    ious, rsna = [], []
    hit = 0
    for (pb, ps), gb, (h, w) in zip(preds, gts, sizes):
        s = rsna_image_score(pb, ps, gb)
        if s is not None:
            rsna.append(s)
        if len(gb):
            iou = union_mask_iou(pb, gb, h, w)
            ious.append(iou)
            hit += int(len(pb) > 0)
    n_pos = len(ious)
    return {"mean_iou": float(np.mean(ious)) if ious else float("nan"),
            "rsna_map": float(np.mean(rsna)) if rsna else float("nan"),
            "frac_pos_with_box": hit / max(n_pos, 1),
            "n_pos_images": n_pos, "n_scored_images": len(rsna),
            "iou_values": [round(v, 4) for v in ious]}

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/models.py
"""Model definitions.

Both classifiers end in   conv feature map -> Global Average Pool -> one Linear
layer, which is exactly the structure Class Activation Mapping (CAM, Zhou et
al. 2016) needs:  M_c(x, y) = sum_k  w_k^c * f_k(x, y).
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class CAMClassifier(nn.Module):
    """Interface: `features(x)` -> (B, C, h, w);  `fc` is Linear(C, 1)."""

    fc: nn.Linear

    def features(self, x: torch.Tensor) -> torch.Tensor:  # pragma: no cover
        raise NotImplementedError

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.features(x)
        return self.fc(F.adaptive_avg_pool2d(f, 1).flatten(1)).squeeze(1)

    @torch.no_grad()
    def cam(self, x: torch.Tensor):
        """Return (logits (B,), CAM maps (B, h, w)) for the pneumonia class."""
        f = self.features(x)
        logits = self.fc(F.adaptive_avg_pool2d(f, 1).flatten(1)).squeeze(1)
        w = self.fc.weight[0]                       # (C,)
        cams = torch.einsum("c,bchw->bhw", w, f)
        return logits, cams


class PaperCNN(CAMClassifier):
    """Re-implementation of the Stanford CS229 model: 10 conv layers, all 3x3,
    zero padding, ReLU, then GAP + a single FC layer.  We add BatchNorm and
    three 2x2 max-pools (the report does not specify down-sampling); at a
    128x128 input the final map is 16x16, which keeps the CAM reasonably sharp.
    """

    def __init__(self, in_ch: int = 1, widths=(32, 32, 64, 64, 128, 128, 256, 256, 256, 256),
                 pool_after=(1, 3, 5)):
        super().__init__()
        layers, c = [], in_ch
        for i, w in enumerate(widths):
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True)]
            if i in pool_after:
                layers.append(nn.MaxPool2d(2))
            c = w
        self.body = nn.Sequential(*layers)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(c, 1)

    def features(self, x):
        return self.body(x)

    def forward(self, x):
        f = self.features(x)
        return self.fc(self.drop(F.adaptive_avg_pool2d(f, 1).flatten(1))).squeeze(1)


class DenseNetCAM(CAMClassifier):
    """DenseNet-121 (the CheXNet backbone), ImageNet-pretrained, with a
    GAP + Linear(1024, 1) head so CAM can be computed directly."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        from torchvision.models import densenet121, DenseNet121_Weights
        net = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        self.backbone = net.features
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(1024, 1)

    def features(self, x):
        return F.relu(self.backbone(x))

    def forward(self, x):
        f = self.features(x)
        return self.fc(self.drop(F.adaptive_avg_pool2d(f, 1).flatten(1))).squeeze(1)


def build_classifier(name: str, pretrained: bool = True) -> CAMClassifier:
    if name == "PaperCNN":
        return PaperCNN()
    if name == "DenseNet121":
        return DenseNetCAM(pretrained=pretrained)
    raise ValueError(name)


def build_detector(pretrained: bool = True, size: int = 512):
    """torchvision Faster R-CNN, ResNet-50-FPN, COCO-pretrained, 2 classes."""
    from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    kw = dict(min_size=size, max_size=size, box_detections_per_img=10)
    if pretrained:
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT, **kw)
    else:
        model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, **kw)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, 2)
    return model


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/pipeline.py
"""End-to-end experiment pipeline.  Each stage reads what earlier stages wrote
to `cache_dir` / `out_dir`, so stages can be re-run on their own:

    prepare    labels -> one row per image, patient split, DICOM cache
    baselines  Logistic Regression / SVM / Random Forest on pixels
    cnn        Paper-style CNN and DenseNet-121 (+ CAMs for val/test)
    detector   Faster R-CNN (supervised boxes)
    localize   CAM -> boxes, threshold tuning on val, IoU + RSNA mAP on test
    figures    all plots in outputs/figures
"""
from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from . import baselines as B
from .cam import cam_predictions, compute_cams
from .config import Config
from .data import (CLASS_OPACITY, CLASSES, ClsDataset, DetDataset, build_cache, build_label_table,
                   load_cache, load_meta, make_split, setting_subset)
from .metrics import (best_f1_threshold, classification_metrics, localization_metrics, roc_points,
                      xywh_to_xyxy)
from .models import build_classifier, build_detector, count_params
from .train import predict_detector, predict_probs, train_classifier, train_detector
from .utils import ensure_dir, get_device, load_json, save_json, seed_everything, timer, update_results

PAPER_REFERENCE = {
    "source": "Huang, Cortes, Monam - Weakly Supervised Pneumonia Localization (CS229, 2018)",
    "classification_test_accuracy": {"LogReg": 0.7302, "SVM": 0.5818, "RandomForest": 0.83,
                                     "PaperCNN": 0.9247},
    "localization_test_iou": {"R-CNN (supervised)": 0.1266, "CNN+CAM (weak)": 0.1508,
                              "CNN+CAM if every positive is localized": 0.379},
    "input_size": 128, "split": "70/20/10", "setting": "Normal vs Lung Opacity (stage-1 data)",
}

CNN_MODELS = ("PaperCNN", "DenseNet121")


def _paths(cfg: Config):
    out = ensure_dir(cfg.out_dir)
    return out, ensure_dir(out / "models"), ensure_dir(out / "preds"), ensure_dir(out / "figures")


def _gt_and_sizes(df: pd.DataFrame):
    gts = [xywh_to_xyxy(b) for b in df["boxes"]]
    sizes = list(zip(df["orig_h"].astype(int), df["orig_w"].astype(int)))
    return gts, sizes


# =============================================================== prepare
def stage_prepare(cfg: Config):
    cache = ensure_dir(cfg.cache_dir)
    out, *_ = _paths(cfg)
    if (cache / "meta.csv").exists() and (cache / f"img{cfg.cache_size}.npy").exists():
        print("cache exists - skipping DICOM decoding")
        df = load_meta(cache)
    else:
        df = build_label_table(cfg.data_dir)
        if cfg.limit:
            df = df.groupby("cls", group_keys=False).apply(
                lambda g: g.sample(min(len(g), max(2, int(cfg.limit * len(g) / len(df)))),
                                   random_state=cfg.seed)).reset_index(drop=True)
        df = make_split(df, cfg.split_fracs, cfg.seed)
        with timer("decode_dicoms", out):
            df = build_cache(df, cfg.data_dir, cache, cfg.cache_size, cfg.num_workers)
    stats = {
        "n_images": int(len(df)),
        "n_boxes": int(df["n_boxes"].sum()),
        "class_counts": df["cls"].value_counts().to_dict(),
        "split_counts": {s: df[df.split == s]["cls"].value_counts().reindex(CLASSES).fillna(0)
                         .astype(int).to_dict() for s in ("train", "val", "test")},
        "boxes_per_positive": df[df.target == 1]["n_boxes"].value_counts().sort_index().to_dict(),
        "image_size": f"{int(df.orig_h.mode()[0])}x{int(df.orig_w.mode()[0])}",
    }
    pos = df[df.target == 1]
    wh = np.array([[b[2], b[3], h * w] for bs, h, w in zip(pos["boxes"], pos["orig_h"], pos["orig_w"])
                   for b in bs]) if len(pos) else np.zeros((0, 3))
    if len(wh):
        stats["box_w_mean"], stats["box_h_mean"] = float(wh[:, 0].mean()), float(wh[:, 1].mean())
        stats["box_area_frac_mean"] = float((wh[:, 0] * wh[:, 1] / wh[:, 2]).mean())
    for s in cfg.settings:
        sub = setting_subset(df, s)
        stats[f"positive_rate_{s}"] = float(sub.target.mean())
        stats[f"n_{s}"] = int(len(sub))
    update_results(out, ["dataset"], stats)
    update_results(out, ["paper_reference"], PAPER_REFERENCE)
    update_results(out, ["config"], cfg.to_dict())
    print(stats)
    return df


# =============================================================== baselines
def stage_baselines(cfg: Config):
    out, *_ = _paths(cfg)
    df = load_meta(cfg.cache_dir)
    arr = load_cache(cfg.cache_dir, cfg.cache_size)
    for setting in cfg.settings:
        sub = setting_subset(df, setting)
        tr, te = sub[sub.split == "train"], sub[sub.split == "test"]
        if len(tr) > cfg.baseline_max_train:
            tr = tr.sample(cfg.baseline_max_train, random_state=cfg.seed)
        with timer(f"baseline_features_{setting}", out):
            Xtr = B.flatten_images(arr, tr["idx"].values, cfg.baseline_size)
            Xte = B.flatten_images(arr, te["idx"].values, cfg.baseline_size)
        n_pca = min(cfg.baseline_pca, Xtr.shape[0] - 1, Xtr.shape[1])
        for name, model in B.build_baselines(n_pca, cfg.seed).items():
            with timer(f"baseline_{name}_{setting}", out):
                model.fit(Xtr, tr["target"].values)
                s_tr, s_te = B.scores(model, Xtr), B.scores(model, Xte)
            m = classification_metrics(te["target"].values, s_te, 0.5)
            m["train_accuracy"] = float(((s_tr >= 0.5) == tr["target"].values).mean())
            m["roc"] = roc_points(te["target"].values, s_te)
            update_results(out, ["classification", setting, name], m)
            print(f"{setting:5s} {name:12s} test acc {m['accuracy']:.4f}  auc {m['auc']:.4f}")


# =============================================================== CNNs
def _cls_datasets(cfg, sub, name):
    if name == "PaperCNN":
        size, three = cfg.paper_cnn_size, False
    else:
        size, three = cfg.densenet_size, True
    mk = lambda d, aug: ClsDataset(d, cfg.cache_dir, cfg.cache_size, size, three, aug)
    return (mk(sub[sub.split == "train"], True), mk(sub[sub.split == "val"], False),
            mk(sub[sub.split == "test"], False))


def stage_cnn(cfg: Config, models=CNN_MODELS):
    out, mdir, pdir, _ = _paths(cfg)
    device = get_device()
    df = load_meta(cfg.cache_dir)
    for setting in cfg.settings:
        sub = setting_subset(df, setting)
        for name in models:
            seed_everything(cfg.seed)
            tr, va, te = _cls_datasets(cfg, sub, name)
            model = build_classifier(name, cfg.pretrained)
            ep, lr, bs = ((cfg.paper_cnn_epochs, cfg.paper_cnn_lr, cfg.paper_cnn_batch) if name == "PaperCNN"
                          else (cfg.densenet_epochs, cfg.densenet_lr, cfg.densenet_batch))
            print(f"== {name} | setting={setting} | train {len(tr)} val {len(va)} test {len(te)} | "
                  f"{count_params(model)/1e6:.2f}M params | device {device}")
            with timer(f"train_{name}_{setting}", out):
                model, hist = train_classifier(model, tr, va, device, ep, lr, bs, cfg.num_workers)
            torch.save(model.state_dict(), mdir / f"{name}_{setting}.pt")
            pv, cv = compute_cams(model, va, device, bs, cfg.num_workers)
            pt, ct = compute_cams(model, te, device, bs, cfg.num_workers)
            ptr = predict_probs(model, tr.__class__(sub[sub.split == "train"], cfg.cache_dir, cfg.cache_size,
                                                    tr.size, tr.three, False), device, bs, cfg.num_workers)
            thr = best_f1_threshold(va.y, pv)
            m = classification_metrics(te.y, pt, thr)
            m["at_0.5"] = {k: v for k, v in classification_metrics(te.y, pt, 0.5).items() if k != "confusion"}
            m["train_accuracy"] = float(((ptr >= thr) == tr.y).mean())
            m["val"] = {k: v for k, v in classification_metrics(va.y, pv, thr).items() if k != "confusion"}
            m["roc"] = roc_points(te.y, pt)
            m["history"] = hist
            m["params_M"] = round(count_params(model) / 1e6, 2)
            m["input_size"] = tr.size
            update_results(out, ["classification", setting, name], m)
            np.savez_compressed(pdir / f"{name}_{setting}.npz",
                                val_idx=sub[sub.split == "val"]["idx"].values, val_prob=pv, val_cam=cv,
                                test_idx=sub[sub.split == "test"]["idx"].values, test_prob=pt, test_cam=ct,
                                cls_threshold=thr)
            print(f"{setting} {name}: test acc {m['accuracy']:.4f} auc {m['auc']:.4f} "
                  f"recall {m['recall']:.4f} (thr {thr:.2f})")
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()


# =============================================================== detector
def stage_detector(cfg: Config):
    out, mdir, pdir, _ = _paths(cfg)
    device = get_device()
    df = load_meta(cfg.cache_dir)
    seed_everything(cfg.seed)
    tr = df[df.split == "train"]
    ds_pos = DetDataset(tr[tr.target == 1], cfg.cache_dir, cfg.cache_size, cfg.det_size, True)
    ds_neg = DetDataset(tr[tr.target == 0], cfg.cache_dir, cfg.cache_size, cfg.det_size, True)
    model = build_detector(cfg.pretrained, cfg.det_size)
    print(f"== Faster R-CNN | pos {len(ds_pos)} neg pool {len(ds_neg)} | {count_params(model)/1e6:.1f}M params")
    with timer("train_detector", out):
        model, hist = train_detector(model, ds_pos, ds_neg, device, cfg.det_epochs, cfg.det_lr,
                                     cfg.det_batch, cfg.det_neg_ratio, cfg.num_workers)
    torch.save(model.state_dict(), mdir / "FasterRCNN.pt")
    preds = {}
    for split in ("val", "test"):
        d = df[df.split == split].reset_index(drop=True)
        with timer(f"detector_predict_{split}", out):
            raw = predict_detector(model, DetDataset(d, cfg.cache_dir, cfg.cache_size, cfg.det_size, False),
                                   device, cfg.det_batch, cfg.num_workers)
        sc = (d["orig_w"].values / cfg.det_size, d["orig_h"].values / cfg.det_size)
        for k, (b, s) in enumerate(raw):
            b = b.copy()
            b[:, [0, 2]] *= sc[0][k]
            b[:, [1, 3]] *= sc[1][k]
            preds[int(d["idx"].iloc[k])] = (b, s)
    with open(pdir / "FasterRCNN.pkl", "wb") as f:
        pickle.dump(preds, f)
    update_results(out, ["detector_training"], {"history": hist, "params_M": round(count_params(model) / 1e6, 1),
                                                "input_size": cfg.det_size, "epochs": cfg.det_epochs})


# =============================================================== localization
def _filter(preds, thr):
    out = []
    for b, s in preds:
        b, s = np.asarray(b).reshape(-1, 4), np.asarray(s)
        k = s >= thr
        out.append((b[k].tolist(), s[k].tolist()))
    return out


def _strip(m):
    return {k: v for k, v in m.items() if k != "iou_values"}


def stage_localize(cfg: Config):
    out, _, pdir, _ = _paths(cfg)
    df = load_meta(cfg.cache_dir).set_index("idx", drop=False)
    det = None
    if (pdir / "FasterRCNN.pkl").exists():
        with open(pdir / "FasterRCNN.pkl", "rb") as f:
            det = pickle.load(f)
    saved = {}
    for setting in cfg.settings:
        res = {}
        for name in CNN_MODELS:
            f = pdir / f"{name}_{setting}.npz"
            if not f.exists():
                continue
            z = np.load(f)
            gate = float(z["cls_threshold"])
            dv, dt = df.loc[z["val_idx"]], df.loc[z["test_idx"]]
            gv, sv = _gt_and_sizes(dv)
            gt, st = _gt_and_sizes(dt)
            # tune the CAM threshold on validation (paper metric: mean IoU, end-to-end)
            sweep = {}
            for thr in cfg.cam_thresholds:
                pv = cam_predictions(z["val_prob"], z["val_cam"], sv, thr, gate, cfg.cam_min_area_frac,
                                     cfg.cam_max_boxes)
                sweep[thr] = _strip(localization_metrics(pv, gv, sv))
            best = max(sweep, key=lambda t: sweep[t]["mean_iou"])
            p_e2e = cam_predictions(z["test_prob"], z["test_cam"], st, best, gate, cfg.cam_min_area_frac,
                                    cfg.cam_max_boxes)
            p_orc = cam_predictions(z["test_prob"], z["test_cam"], st, best, None, cfg.cam_min_area_frac,
                                    cfg.cam_max_boxes)
            e2e, orc = localization_metrics(p_e2e, gt, st), localization_metrics(p_orc, gt, st)
            res[f"{name}+CAM"] = {"type": "weakly supervised", "cam_threshold": best, "cls_gate": gate,
                                  "val_sweep": {str(k): v for k, v in sweep.items()},
                                  "test": _strip(e2e), "test_oracle": _strip(orc),
                                  "test_iou_values": e2e["iou_values"]}
            saved[(setting, f"{name}+CAM")] = dict(zip(z["test_idx"].tolist(), p_e2e))
            print(f"{setting} {name}+CAM thr {best}: mIoU {e2e['mean_iou']:.4f} (oracle {orc['mean_iou']:.4f})"
                  f" RSNA mAP {e2e['rsna_map']:.4f}")

        if det is not None:
            sub = setting_subset(df, setting)
            dv, dt = sub[sub.split == "val"], sub[sub.split == "test"]
            gv, sv = _gt_and_sizes(dv)
            gt, st = _gt_and_sizes(dt)
            rv = [det[i] for i in dv["idx"]]
            rt = [det[i] for i in dt["idx"]]
            sweep = {t: _strip(localization_metrics(_filter(rv, t), gv, sv)) for t in cfg.det_score_thresholds}
            best = max(sweep, key=lambda t: sweep[t]["rsna_map"])
            p_det = _filter(rt, best)
            m = localization_metrics(p_det, gt, st)
            res["FasterRCNN"] = {"type": "supervised", "score_threshold": best,
                                 "val_sweep": {str(k): v for k, v in sweep.items()},
                                 "test": _strip(m), "test_iou_values": m["iou_values"]}
            saved[(setting, "FasterRCNN")] = dict(zip(dt["idx"].tolist(), p_det))
            print(f"{setting} FasterRCNN score>={best}: mIoU {m['mean_iou']:.4f} RSNA mAP {m['rsna_map']:.4f}")

            # classifier-gated detector: keep boxes only if DenseNet says "pneumonia"
            f = pdir / f"DenseNet121_{setting}.npz"
            if f.exists():
                z = np.load(f)
                prob_v = dict(zip(z["val_idx"].tolist(), z["val_prob"]))
                prob_t = dict(zip(z["test_idx"].tolist(), z["test_prob"]))
                best_m, best_cfg = -1, None
                for g in cfg.gate_thresholds:
                    for t in cfg.det_score_thresholds:
                        pv = [(b, s) if prob_v[i] >= g else ([], []) for i, (b, s) in
                              zip(dv["idx"], _filter(rv, t))]
                        mv = localization_metrics(pv, gv, sv)["rsna_map"]
                        if mv > best_m:
                            best_m, best_cfg = mv, (g, t)
                g, t = best_cfg
                pt = [(b, s) if prob_t[i] >= g else ([], []) for i, (b, s) in zip(dt["idx"], _filter(rt, t))]
                m = localization_metrics(pt, gt, st)
                res["DenseNet-gated FasterRCNN"] = {"type": "supervised + classifier gate", "gate": g,
                                                    "score_threshold": t, "val_rsna_map": best_m,
                                                    "test": _strip(m), "test_iou_values": m["iou_values"]}
                saved[(setting, "DenseNet-gated FasterRCNN")] = dict(zip(dt["idx"].tolist(), pt))
                print(f"{setting} gated detector (gate {g}, score {t}): mIoU {m['mean_iou']:.4f} "
                      f"RSNA mAP {m['rsna_map']:.4f}")
        update_results(out, ["localization", setting], res)
    with open(pdir / "localization_test_predictions.pkl", "wb") as f:
        pickle.dump(saved, f)


# =============================================================== demo samples
def stage_demo(cfg: Config, n_pos: int = 6, n_neg: int = 3):
    """Export a few *test-set* images as PNG (+ ground truth) for the live demo."""
    import cv2
    import json
    out, *_ = _paths(cfg)
    ddir = ensure_dir(out / "demo_samples")
    df = load_meta(cfg.cache_dir)
    arr = load_cache(cfg.cache_dir, cfg.cache_size)
    te = df[df.split == "test"]
    pick = pd.concat([te[te.target == 1].sample(min(n_pos, (te.target == 1).sum()), random_state=7),
                      te[te.cls == CLASSES[0]].sample(min(n_neg, (te.cls == CLASSES[0]).sum()), random_state=7),
                      te[te.cls == CLASSES[1]].sample(min(n_neg, (te.cls == CLASSES[1]).sum()), random_state=7)])
    gt = {}
    for _, r in pick.iterrows():
        name = f"{'pneumonia' if r.target else 'negative'}_{r.patientId}.png"
        cv2.imwrite(str(ddir / name), np.asarray(arr[r["idx"]]))
        s = cfg.cache_size / r["orig_w"]
        gt[name] = {"cls": r["cls"], "boxes": [[x * s, y * s, (x + w) * s, (y + h) * s] for x, y, w, h in r["boxes"]]}
    (ddir / "ground_truth.json").write_text(json.dumps(gt, indent=1))
    print(f"exported {len(gt)} demo images to {ddir}")


# =============================================================== all
STAGES = {"prepare": stage_prepare, "baselines": stage_baselines, "cnn": stage_cnn,
          "detector": stage_detector, "localize": stage_localize, "demo": stage_demo}


def run(cfg: Config, stages):
    seed_everything(cfg.seed)
    out = ensure_dir(cfg.out_dir)
    cfg.save(out / "config.json")
    for s in stages:
        if s == "detector" and not cfg.run_detector:
            continue
        if s == "figures":
            from .viz import make_all_figures
            with timer("figures"):
                make_all_figures(cfg)
            continue
        with timer(f"stage_{s}", out):
            STAGES[s](cfg)

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/train.py
"""Training / inference loops for the CNN classifiers and the detector."""
from __future__ import annotations

import copy
import math
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from .data import det_collate


def _amp(device):
    return device.type == "cuda"


@torch.no_grad()
def predict_probs(model, ds, device, batch: int = 64, workers: int = 2) -> np.ndarray:
    model.eval()
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers)
    out = []
    for x, _ in dl:
        with torch.autocast(device.type, enabled=_amp(device)):
            out.append(torch.sigmoid(model(x.to(device, non_blocking=True)).float()).cpu())
    return torch.cat(out).numpy()


def train_classifier(model, train_ds, val_ds, device, epochs: int, lr: float, batch: int,
                     workers: int = 4, weight_decay: float = 1e-4, log=print):
    """Adam(W) + cosine schedule + BCE (pos_weight balances classes).
    Keeps the weights of the epoch with the best validation ROC-AUC."""
    model.to(device)
    y = train_ds.y
    pos_weight = torch.tensor([(1 - y.mean()) / max(y.mean(), 1e-6)], device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    dl = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=workers,
                    drop_last=len(train_ds) > batch, pin_memory=device.type == "cuda",
                    persistent_workers=workers > 0)
    total = epochs * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, len(dl) // 2)) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler(enabled=_amp(device))
    hist = {"epoch": [], "train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}
    best_key, best_state = None, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, losses = time.time(), []
        for x, t in dl:
            x, t = x.to(device, non_blocking=True), t.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=_amp(device)):
                loss = crit(model(x).float(), t)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
        pv = predict_probs(model, val_ds, device, batch=batch * 2, workers=workers)
        yv = val_ds.y
        eps = 1e-6
        vloss = float(-np.mean(yv * np.log(pv + eps) + (1 - yv) * np.log(1 - pv + eps)))
        auc = roc_auc_score(yv, pv) if len(np.unique(yv)) > 1 else float("nan")
        acc = float(((pv >= 0.5) == yv).mean())
        for k, v in zip(hist, [ep, float(np.mean(losses)), vloss, float(auc), acc]):
            hist[k].append(v)
        log(f"  epoch {ep}/{epochs}  train_loss {np.mean(losses):.4f}  val_loss {vloss:.4f}  "
            f"val_auc {auc:.4f}  val_acc {acc:.4f}  ({time.time()-t0:.0f}s)")
        # select by validation AUC, ties broken by lower validation loss
        key = (round(float(np.nan_to_num(auc, nan=0.0)), 4), -vloss)
        if best_key is None or key > best_key:
            best_key, best_state = key, copy.deepcopy(model.state_dict())
            hist["best_epoch"] = ep
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist


# ------------------------------------------------------------------ detector
def train_detector(model, ds_pos, ds_neg, device, epochs: int, lr: float, batch: int,
                   neg_ratio: float = 1.0, workers: int = 4, log=print):
    """Faster R-CNN fine-tuning. Each epoch uses every positive image plus a
    fresh random sample of `neg_ratio * n_pos` negatives (empty targets teach
    the detector to stay silent on healthy lungs)."""
    from torch.utils.data import ConcatDataset, Subset
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    n_neg = min(len(ds_neg), int(neg_ratio * len(ds_pos)))
    steps_per_epoch = math.ceil((len(ds_pos) + n_neg) / batch)
    total = epochs * steps_per_epoch
    warm = min(300, steps_per_epoch)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler(enabled=_amp(device))
    hist = {"epoch": [], "train_loss": []}
    rng = np.random.default_rng(0)
    for ep in range(1, epochs + 1):
        model.train()
        neg_idx = rng.choice(len(ds_neg), n_neg, replace=False) if n_neg else []
        ds = ConcatDataset([ds_pos, Subset(ds_neg, list(neg_idx))]) if n_neg else ds_pos
        dl = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=workers,
                        collate_fn=det_collate, pin_memory=device.type == "cuda")
        t0, losses = time.time(), []
        for imgs, targets, _ in dl:
            imgs = [i.to(device, non_blocking=True) for i in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device.type, enabled=_amp(device)):
                loss_dict = model(imgs, targets)
                loss = sum(loss_dict.values())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
        hist["epoch"].append(ep)
        hist["train_loss"].append(float(np.mean(losses)))
        log(f"  det epoch {ep}/{epochs}  loss {np.mean(losses):.4f}  ({time.time()-t0:.0f}s)")
    return model, hist


@torch.no_grad()
def predict_detector(model, ds, device, batch: int = 8, workers: int = 4):
    """Returns per image (boxes [N,4] in *dataset pixels*, scores [N])."""
    model.eval()
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers, collate_fn=det_collate)
    out = [None] * len(ds)
    for imgs, _, ids in dl:
        imgs = [i.to(device) for i in imgs]
        with torch.autocast(device.type, enabled=_amp(device)):
            res = model(imgs)
        for r, i in zip(res, ids):
            out[i] = (r["boxes"].float().cpu().numpy(), r["scores"].float().cpu().numpy())
    return out

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/utils.py
"""Small helpers shared by all stages."""
from __future__ import annotations

import json
import os
import random
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def get_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def ensure_dir(p: str | Path) -> Path:
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def _to_jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def save_json(obj, path: str | Path) -> None:
    Path(path).write_text(json.dumps(obj, indent=2, default=_to_jsonable))


def load_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    return json.loads(path.read_text())


def update_results(out_dir: str | Path, keys: list[str], value) -> None:
    """Write `value` into outputs/results.json at the nested location `keys`.

    Every stage writes its own part, so stages can be re-run independently.
    """
    path = Path(out_dir) / "results.json"
    res = load_json(path)
    d = res
    for k in keys[:-1]:
        d = d.setdefault(k, {})
    d[keys[-1]] = value
    save_json(res, path)


@contextmanager
def timer(name: str, out_dir: str | Path | None = None):
    t0 = time.time()
    print(f"[{name}] started", flush=True)
    yield
    dt = time.time() - t0
    print(f"[{name}] finished in {dt/60:.1f} min", flush=True)
    if out_dir is not None:
        update_results(out_dir, ["timings_min", name], round(dt / 60, 2))

In [ ]:
%%writefile /kaggle/working/repo/src/pneumo/viz.py
"""All figures for the write-up and the slides (saved to outputs/figures)."""
from __future__ import annotations

import pickle
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

from .cam import upsample
from .config import Config
from .data import CLASSES, load_cache, load_meta
from .utils import ensure_dir, load_json

GT_C, CAM_C, DET_C = "#e3342f", "#2f6fe3", "#18a558"
MODEL_COLORS = {"LogReg": "#9aa0a6", "SVM": "#c49a6c", "RandomForest": "#8e6bbf",
                "PaperCNN": "#f29d38", "DenseNet121": "#2f6fe3"}
LABELS = {"LogReg": "Logistic Regression", "SVM": "SVM (RBF)", "RandomForest": "Random Forest",
          "PaperCNN": "Paper-style CNN (128px)", "DenseNet121": "DenseNet-121 (pretrained)"}

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 150, "savefig.bbox": "tight"})


def _draw_boxes(ax, boxes, color, scale=1.0, lw=1.8, ls="-"):
    for x1, y1, x2, y2 in boxes:
        ax.add_patch(patches.Rectangle((x1 * scale, y1 * scale), (x2 - x1) * scale, (y2 - y1) * scale,
                                       fill=False, edgecolor=color, linewidth=lw, linestyle=ls))


def fig_data_overview(cfg, df, arr, fdir):
    fig = plt.figure(figsize=(11, 3.6))
    gs = fig.add_gridspec(1, 4, width_ratios=[1.3, 1, 1, 1], wspace=0.15)
    ax = fig.add_subplot(gs[0])
    counts = df["cls"].value_counts().reindex(CLASSES).fillna(0)
    ax.barh(["Normal", "Not normal,\nno opacity", "Lung opacity\n(pneumonia)"], counts.values,
            color=["#9aa0a6", "#c49a6c", GT_C])
    for i, v in enumerate(counts.values):
        ax.text(v, i, f" {int(v):,}", va="center", fontsize=9)
    ax.set_xlabel("images")
    ax.set_title("RSNA training images by class")
    ax.set_xlim(0, counts.max() * 1.25)
    rng = np.random.default_rng(1)
    s = cfg.cache_size
    for j, c in enumerate(CLASSES):
        a = fig.add_subplot(gs[j + 1])
        cand = df[df.cls == c]
        if c == CLASSES[2]:
            cand = cand[cand.n_boxes >= 2] if (cand.n_boxes >= 2).any() else cand
        if not len(cand):
            a.axis("off")
            continue
        r = cand.iloc[rng.integers(len(cand))]
        a.imshow(arr[r["idx"]], cmap="gray")
        _draw_boxes(a, [[x, y, x + w, y + h] for x, y, w, h in r["boxes"]], GT_C, s / r["orig_w"])
        a.set_title(["Normal", "Not normal / no opacity", "Lung opacity + boxes"][j], fontsize=9)
        a.axis("off")
    fig.savefig(fdir / "data_overview.png")
    plt.close(fig)


def fig_roc(res, setting, fdir):
    cl = res.get("classification", {}).get(setting, {})
    if not cl:
        return
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    for name in ["LogReg", "SVM", "RandomForest", "PaperCNN", "DenseNet121"]:
        if name in cl and "roc" in cl[name]:
            fpr, tpr = cl[name]["roc"]
            ax.plot(fpr, tpr, color=MODEL_COLORS[name], lw=2 if "CNN" in name or "Dense" in name else 1.4,
                    label=f"{LABELS[name]}  AUC {cl[name]['auc']:.3f}")
    ax.plot([0, 1], [0, 1], ":", color="#bbb", lw=1)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title({"paper": "ROC - Normal vs Pneumonia (paper setting)",
                  "full": "ROC - Pneumonia vs all (full dataset)"}[setting], fontsize=10)
    ax.legend(fontsize=7.5, loc="lower right", frameon=False)
    fig.savefig(fdir / f"roc_{setting}.png")
    plt.close(fig)


def fig_training(res, fdir):
    cl = res.get("classification", {})
    runs = [(s, n) for s in cl for n in ("PaperCNN", "DenseNet121") if n in cl[s] and "history" in cl[s][n]]
    if not runs:
        return
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
    for s, n in runs:
        h = cl[s][n]["history"]
        ls = "-" if s == "paper" else "--"
        lab = f"{LABELS[n].split(' (')[0]} - {s}"
        axes[0].plot(h["epoch"], h["train_loss"], ls, color=MODEL_COLORS[n], label=lab)
        axes[1].plot(h["epoch"], h["val_auc"], ls, color=MODEL_COLORS[n], marker="o", ms=3, label=lab)
    axes[0].set_title("Training loss (weighted BCE)")
    axes[1].set_title("Validation ROC-AUC")
    for a in axes:
        a.set_xlabel("epoch")
    axes[1].legend(fontsize=7.5, frameon=False)
    fig.savefig(fdir / "training_curves.png")
    plt.close(fig)


def fig_confusion(res, setting, name, fdir):
    m = res.get("classification", {}).get(setting, {}).get(name)
    if not m:
        return
    c = m["confusion"]
    mat = np.array([[c["tn"], c["fp"]], [c["fn"], c["tp"]]])
    fig, ax = plt.subplots(figsize=(3.2, 3))
    ax.imshow(mat, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{mat[i, j]:,}", ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() / 2 else "black", fontsize=11)
    ax.set_xticks([0, 1], ["pred. no", "pred. pneumonia"])
    ax.set_yticks([0, 1], ["no pneumonia", "pneumonia"])
    ax.set_title(f"{LABELS[name].split(' (')[0]} - test ({setting})", fontsize=9)
    fig.savefig(fdir / f"confusion_{setting}_{name}.png")
    plt.close(fig)


def fig_paper_comparison(res, fdir):
    ref = res.get("paper_reference", {})
    cl = res.get("classification", {}).get("paper", {})
    loc = res.get("localization", {}).get("paper", {})
    if not cl:
        return
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1.25, 1]})
    names = [n for n in ["LogReg", "SVM", "RandomForest", "PaperCNN", "DenseNet121"] if n in cl]
    x = np.arange(len(names))
    paper = [ref.get("classification_test_accuracy", {}).get(n, np.nan) for n in names]
    ours = [cl[n]["accuracy"] for n in names]
    axes[0].bar(x - 0.2, paper, 0.4, color="#cfd3d8", label="Stanford report")
    axes[0].bar(x + 0.2, ours, 0.4, color=[MODEL_COLORS[n] for n in names], label="ours")
    for xi, v in zip(x, ours):
        axes[0].text(xi + 0.2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
    for xi, v in zip(x, paper):
        if not np.isnan(v):
            axes[0].text(xi - 0.2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8, color="#777")
    axes[0].set_xticks(x, [LABELS[n].split(" (")[0].replace(" ", "\n", 1) for n in names], fontsize=8)
    axes[0].set_ylim(0, 1.18)
    axes[0].set_ylabel("test accuracy")
    axes[0].set_title("Classification (Normal vs Pneumonia)")
    axes[0].legend(fontsize=8, frameon=False, loc="upper left", ncol=2)
    lab, val, col = ["Report\nR-CNN", "Report\nCNN\n+CAM"], [0.1266, 0.1508], ["#cfd3d8", "#cfd3d8"]
    for k, short, c in [("PaperCNN+CAM", "Ours\nCNN\n+CAM", MODEL_COLORS["PaperCNN"]),
                        ("DenseNet121+CAM", "Ours\nDenseNet\n+CAM", MODEL_COLORS["DenseNet121"]),
                        ("FasterRCNN", "Ours\nFaster\nR-CNN", DET_C),
                        ("DenseNet-gated FasterRCNN", "Ours\ngated\nF-RCNN", "#0d6b3a")]:
        if k in loc:
            lab.append(short)
            val.append(loc[k]["test"]["mean_iou"])
            col.append(c)
    axes[1].bar(range(len(val)), val, color=col)
    for i, v in enumerate(val):
        axes[1].text(i, v + 0.008, f"{v:.3f}", ha="center", fontsize=8)
    axes[1].set_xticks(range(len(val)), lab, fontsize=7.5)
    axes[1].set_ylabel("mean IoU on pneumonia test images")
    axes[1].set_ylim(0, max(val) * 1.2 + 1e-3)
    axes[1].set_title("Localization")
    fig.savefig(fdir / "paper_comparison.png")
    plt.close(fig)


def fig_localization_examples(cfg, df, arr, res, fdir, setting="paper", n=6):
    pdir = Path(cfg.out_dir) / "preds"
    f = pdir / f"DenseNet121_{setting}.npz"
    lp = pdir / "localization_test_predictions.pkl"
    if not f.exists() or not lp.exists():
        return
    z = np.load(f)
    with open(lp, "rb") as fh:
        saved = pickle.load(fh)
    cam_p = saved.get((setting, "DenseNet121+CAM"), {})
    det_p = saved.get((setting, "DenseNet-gated FasterRCNN"), saved.get((setting, "FasterRCNN"), {}))
    pos = [k for k, i in enumerate(z["test_idx"]) if df.loc[i, "target"] == 1]
    rng = np.random.default_rng(3)
    pick = rng.choice(pos, min(n, len(pos)), replace=False) if pos else []
    if not len(pick):
        return
    s = cfg.cache_size
    fig, axes = plt.subplots(len(pick), 3, figsize=(7.5, 2.55 * len(pick)))
    axes = np.atleast_2d(axes)
    for r, k in enumerate(pick):
        i = int(z["test_idx"][k])
        row = df.loc[i]
        img = arr[i]
        sc = s / row["orig_w"]
        gt = [[x, y, x + w, y + h] for x, y, w, h in row["boxes"]]
        cam = np.maximum(upsample(z["test_cam"][k], s, s), 0)
        cam = cam / cam.max() if cam.max() > 0 else cam
        axes[r, 0].imshow(img, cmap="gray")
        _draw_boxes(axes[r, 0], gt, GT_C, sc)
        axes[r, 1].imshow(img, cmap="gray")
        axes[r, 1].imshow(cam, cmap="jet", alpha=0.38)
        _draw_boxes(axes[r, 1], gt, GT_C, sc, lw=1.2, ls="--")
        _draw_boxes(axes[r, 1], cam_p.get(i, ([], []))[0], CAM_C, sc, lw=2)
        axes[r, 2].imshow(img, cmap="gray")
        _draw_boxes(axes[r, 2], gt, GT_C, sc, lw=1.2, ls="--")
        db, ds = det_p.get(i, ([], []))
        _draw_boxes(axes[r, 2], db, DET_C, sc, lw=2)
        for b, sco in zip(db, ds):
            axes[r, 2].text(b[0] * sc, b[1] * sc - 4, f"{sco:.2f}", color=DET_C, fontsize=7)
        axes[r, 1].set_xlabel(f"p(pneumonia) = {z['test_prob'][k]:.2f}", fontsize=8)
        for a in axes[r]:
            a.set_xticks([])
            a.set_yticks([])
    axes[0, 0].set_title("X-ray + ground truth", fontsize=9, color=GT_C)
    axes[0, 1].set_title("DenseNet CAM -> boxes (weak)", fontsize=9, color=CAM_C)
    axes[0, 2].set_title("Faster R-CNN (supervised)", fontsize=9, color=DET_C)
    fig.tight_layout()
    fig.savefig(fdir / f"localization_examples_{setting}.png")
    plt.close(fig)


def fig_iou_hist(res, setting, fdir):
    loc = res.get("localization", {}).get(setting, {})
    keys = [k for k in ["DenseNet121+CAM", "FasterRCNN", "DenseNet-gated FasterRCNN"] if k in loc]
    if not keys:
        return
    fig, ax = plt.subplots(figsize=(5, 3.2))
    cols = {"DenseNet121+CAM": CAM_C, "FasterRCNN": DET_C, "DenseNet-gated FasterRCNN": "#0d6b3a"}
    bins = np.linspace(0, 1, 21)
    for k in keys:
        ax.hist(loc[k]["test_iou_values"], bins=bins, histtype="step", lw=1.8, color=cols[k], label=k)
    ax.set_xlabel("IoU with ground truth (per pneumonia test image)")
    ax.set_ylabel("images")
    ax.set_title("IoU distribution (0 = missed or wrong place)", fontsize=10)
    ax.legend(fontsize=7.5, frameon=False)
    fig.savefig(fdir / f"iou_hist_{setting}.png")
    plt.close(fig)


def fig_cam_sweep(res, setting, fdir):
    loc = res.get("localization", {}).get(setting, {})
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    found = False
    for k, c in [("PaperCNN+CAM", MODEL_COLORS["PaperCNN"]), ("DenseNet121+CAM", MODEL_COLORS["DenseNet121"])]:
        if k in loc:
            sw = loc[k]["val_sweep"]
            t = sorted(sw, key=float)
            ax.plot([float(x) for x in t], [sw[x]["mean_iou"] for x in t], "o-", color=c, label=k)
            found = True
    if not found:
        plt.close(fig)
        return
    ax.set_xlabel("CAM threshold (fraction of map maximum)")
    ax.set_ylabel("validation mean IoU")
    ax.set_title("Tuning the heat-map cut-off", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    fig.savefig(fdir / f"cam_threshold_sweep_{setting}.png")
    plt.close(fig)


def make_all_figures(cfg: Config):
    fdir = ensure_dir(Path(cfg.out_dir) / "figures")
    res = load_json(Path(cfg.out_dir) / "results.json")
    df = load_meta(cfg.cache_dir).set_index("idx", drop=False)
    arr = load_cache(cfg.cache_dir, cfg.cache_size)
    fig_data_overview(cfg, df, arr, fdir)
    fig_training(res, fdir)
    fig_paper_comparison(res, fdir)
    for s in cfg.settings:
        fig_roc(res, s, fdir)
        fig_confusion(res, s, "DenseNet121", fdir)
        fig_iou_hist(res, s, fdir)
        fig_cam_sweep(res, s, fdir)
        fig_localization_examples(cfg, df, arr, res, fdir, s)
    print("figures written to", fdir)

In [ ]:
%%writefile /kaggle/working/repo/scripts/run_all.py
"""Run the whole experiment (or selected stages).

Examples
--------
Full run on Kaggle (GPU):
    python scripts/run_all.py --data-dir /kaggle/input/rsna-pneumonia-detection-challenge \
        --cache-dir /tmp/cache --out-dir /kaggle/working/outputs

Quicker run (~45 min, fewer epochs):
    python scripts/run_all.py --fast ...

Only re-do localization + figures after training:
    python scripts/run_all.py --stages localize,figures ...

CPU smoke test on synthetic data (no download needed):
    python scripts/run_all.py --smoke
"""
import argparse
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

from pneumo.config import Config  # noqa: E402
from pneumo.pipeline import run  # noqa: E402

ALL = ["prepare", "baselines", "cnn", "detector", "localize", "figures", "demo"]


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data-dir")
    ap.add_argument("--cache-dir")
    ap.add_argument("--out-dir")
    ap.add_argument("--stages", default=",".join(ALL))
    ap.add_argument("--fast", action="store_true", help="fewer epochs (~45 min on a T4)")
    ap.add_argument("--smoke", action="store_true", help="tiny synthetic CPU run")
    ap.add_argument("--limit", type=int, help="use only N images")
    ap.add_argument("--no-detector", action="store_true")
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--workers", type=int)
    a = ap.parse_args()

    if a.smoke:
        syn = ROOT / "synthetic_rsna"
        if not (syn / "stage_2_train_labels.csv").exists():
            subprocess.check_call([sys.executable, str(ROOT / "scripts" / "make_synthetic_data.py"),
                                   "--out", str(syn)])
        cfg = Config.smoke(data_dir=str(syn), cache_dir=str(ROOT / "cache_smoke"),
                           out_dir=str(ROOT / "outputs_smoke"))
    else:
        cfg = Config.fast() if a.fast else Config()
    for k, v in [("data_dir", a.data_dir), ("cache_dir", a.cache_dir), ("out_dir", a.out_dir),
                 ("limit", a.limit), ("num_workers", a.workers)]:
        if v is not None:
            setattr(cfg, k, v)
    if a.no_detector:
        cfg.run_detector = False
    if a.no_pretrained:
        cfg.pretrained = False
    run(cfg, [s.strip() for s in a.stages.split(",") if s.strip()])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/app.py
"""Live demo (Gradio).  Upload a chest X-ray (DICOM / PNG / JPG) and get:
  * the probability of pneumonia from DenseNet-121,
  * the Class Activation Map heat-map with weakly-supervised boxes (blue),
  * Faster R-CNN boxes (green), and the ground truth (red) for the bundled
    test examples.

    python app.py                       # http://127.0.0.1:7860
    python app.py --share               # public link (e.g. when running on Kaggle/Colab)
"""
import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

import gradio as gr  # noqa: E402

from pneumo.data import read_any_image  # noqa: E402
from pneumo.infer import PneumoniaPredictor, render  # noqa: E402

_cache = {}


def get_predictor(out_dir, setting):
    key = (out_dir, setting)
    if key not in _cache:
        _cache[key] = PneumoniaPredictor(out_dir, setting)
    return _cache[key]


def build(out_dir: str):
    samples_dir = Path(out_dir) / "demo_samples"
    gt = json.loads((samples_dir / "ground_truth.json").read_text()) if (samples_dir / "ground_truth.json").exists() else {}
    examples = sorted(str(p) for p in samples_dir.glob("*.png"))

    def run(file, setting):
        if file is None:
            return None, "Upload an X-ray first."
        path = file if isinstance(file, str) else file.name
        img = read_any_image(path)
        pred = get_predictor(out_dir, setting)(img)
        g = gt.get(Path(path).name, {})
        vis = render(img, pred, g.get("boxes"))
        lines = [f"### p(pneumonia) = **{pred['prob']:.3f}**  "
                 f"(threshold {pred['cls_threshold']:.2f}) → "
                 f"**{'PNEUMONIA' if pred['positive'] else 'no pneumonia'}**",
                 f"- CAM boxes (blue, weakly supervised): {len(pred['cam_boxes'])}",
                 f"- Faster R-CNN boxes (green, supervised): {len(pred['det_boxes'])}"]
        if g:
            lines.append(f"- Ground truth (red): **{g.get('cls', '?')}**, {len(g.get('boxes', []))} box(es)")
        return vis, "\n".join(lines)

    with gr.Blocks(title="Pneumonia detection & localization") as demo:
        gr.Markdown("## Pneumonia Diagnosis, Detection & Localization on Chest X-rays\n"
                    "DenseNet-121 classifier + Class Activation Map (weakly supervised) and "
                    "Faster R-CNN (supervised). Trained on the RSNA Pneumonia Detection Challenge data. "
                    "*Educational project - not a medical device.*")
        with gr.Row():
            with gr.Column(scale=1):
                f = gr.File(label="Chest X-ray (.dcm / .png / .jpg)", file_types=[".dcm", ".png", ".jpg", ".jpeg"],
                            type="filepath")
                setting = gr.Radio(["full", "paper"], value="full", label="Model trained on",
                                   info="full = all RSNA images; paper = Normal vs Pneumonia only")
                btn = gr.Button("Analyse", variant="primary")
                if examples:
                    gr.Examples(examples=[[e] for e in examples], inputs=[f], label="Test-set examples")
            with gr.Column(scale=2):
                out_img = gr.Image(label="Heat-map + boxes", type="numpy")
                out_txt = gr.Markdown()
        btn.click(run, [f, setting], [out_img, out_txt])
        f.change(run, [f, setting], [out_img, out_txt])
    return demo


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", default=str(ROOT / "outputs"))
    ap.add_argument("--share", action="store_true")
    ap.add_argument("--port", type=int, default=7860)
    a = ap.parse_args()
    build(a.out_dir).launch(share=a.share, server_port=a.port)

In [ ]:
%%writefile /kaggle/working/repo/predict.py
"""Command-line prediction on one or more chest X-rays (DICOM / PNG / JPG).

    python predict.py path/to/xray.dcm [more files] --out-dir outputs --save-dir predictions
"""
import argparse
import sys
from pathlib import Path

import cv2

sys.path.insert(0, str(Path(__file__).resolve().parent / "src"))
from pneumo.data import read_any_image  # noqa: E402
from pneumo.infer import PneumoniaPredictor, render  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("images", nargs="+")
    ap.add_argument("--out-dir", default="outputs", help="folder with results.json and models/")
    ap.add_argument("--setting", default="full", choices=["full", "paper"])
    ap.add_argument("--save-dir", default="predictions")
    a = ap.parse_args()
    pred = PneumoniaPredictor(a.out_dir, a.setting)
    Path(a.save_dir).mkdir(exist_ok=True)
    for p in a.images:
        img = read_any_image(p)
        r = pred(img)
        verdict = "PNEUMONIA" if r["positive"] else "no pneumonia"
        print(f"{p}: p(pneumonia)={r['prob']:.3f} -> {verdict}; "
              f"CAM boxes {[[round(v) for v in b] for b in r['cam_boxes']]}; "
              f"detector boxes {[[round(v) for v in b] for b in r['det_boxes']]}")
        out = Path(a.save_dir) / (Path(p).stem + "_pred.png")
        cv2.imwrite(str(out), render(img, r)[:, :, ::-1])
    print("visualisations saved to", a.save_dir)


if __name__ == "__main__":
    main()

## 2 · Prepare data
Collapse box rows to one row per image, stratified patient-level 70/15/15 split, decode all 26,684 DICOMs once into a 512×512 uint8 memory-map (~5 min).

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages prepare

## 3 · Classical baselines (LR / SVM / RF on 64×64 pixels)

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages baselines

## 4 · CNN classifiers: paper-style CNN (128 px) and DenseNet-121 (pretrained)
Trained in both settings: **paper** (Normal vs Lung Opacity) and **full** (Lung Opacity vs everything).

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages cnn

## 5 · Supervised detector: Faster R-CNN ResNet-50-FPN (COCO-pretrained)

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages detector

## 6 · Localization: CAM → boxes, threshold tuning on validation, mean IoU + RSNA mAP on test

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages localize

## 7 · Figures and demo samples

In [ ]:
!cd /kaggle/working/repo && python scripts/run_all.py --data-dir $DATA_DIR --cache-dir /tmp/cache --out-dir /kaggle/working/outputs $MODE --stages figures,demo

In [ ]:
import json, pandas as pd
R = json.load(open("/kaggle/working/outputs/results.json"))
for s in R["classification"]:
    print(f"\n=== classification - {s} setting (test) ===")
    print(pd.DataFrame({k: {m: v[m] for m in ["accuracy", "precision", "recall", "specificity", "f1", "auc"]}
                        for k, v in R["classification"][s].items()}).T.round(4))
for s in R.get("localization", {}):
    print(f"\n=== localization - {s} setting (test) ===")
    print(pd.DataFrame({k: v["test"] for k, v in R["localization"][s].items()}).T
          [["mean_iou", "rsna_map", "frac_pos_with_box"]].round(4))
print("\ntimings (min):", R.get("timings_min"))

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/kaggle/working/outputs/figures/*.png")):
    print(f); display(Image(f, width=900))

## 8 · Package the outputs
`results_light.zip` (numbers + figures, small) → upload this to Claude / use for the report.  
`model_weights.zip` → put its `models/` folder inside `outputs/` of the repo to run the live demo.

In [ ]:
%cd /kaggle/working
!cd outputs && zip -qr ../results_light.zip results.json config.json figures demo_samples
!cd outputs && zip -qr ../model_weights.zip models
!ls -lh /kaggle/working/*.zip

### Optional: live demo inside Kaggle
Run the cell below for a public Gradio link (needs Internet on).

In [ ]:
# !pip -q install gradio
# %cd /kaggle/working/repo
# import sys; sys.argv = ["app.py", "--out-dir", "/kaggle/working/outputs", "--share"]
# exec(open("app.py").read())